In [3]:
from pathlib import Path
pp = Path('../data/processed/')
if pp.exists():
    print([f.name for f in pp.glob('*')])
else:
    print("Le dossier data/processed/ n'existe pas")

['confusion_matrix_cmapss.png', 'confusion_matrix_cwru.png', 'confusion_matrix_mcc5_v3.png', 'confusion_matrix_mf_v2.png', 'confusion_matrix_vbl.png', 'meta_cwru.pkl', 'meta_mf.pkl', 'rul_prediction_cmapss.png', 'X_cwru.npy', 'X_mf.npy', 'y_cwru.pkl', 'y_mf.pkl']


In [4]:
import sys
sys.path.append('../')

import numpy as np
import pickle
from pathlib import Path

def load_dataset(name):
    X    = np.load(f'../data/processed/X_{name}.npy')
    with open(f'../data/processed/y_{name}.pkl', 'rb') as f:
        y = pickle.load(f)
    with open(f'../data/processed/meta_{name}.pkl', 'rb') as f:
        meta = pickle.load(f)
    return X, y, meta

# Chargement
X_vbl,   y_vbl,   meta_vbl   = load_dataset('vbl')
X_cwru,  y_cwru,  meta_cwru  = load_dataset('cwru')
X_cmapss,y_cmapss,meta_cmapss= load_dataset('cmapss')
X_mf,    y_mf,    meta_mf    = load_dataset('mf')

print("Datasets chargés :")
print(f"  VBL-VA001        : {X_vbl.shape}")
print(f"  CWRU             : {X_cwru.shape}")
print(f"  CMAPSS           : {X_cmapss.shape}")
print(f"  Mechanical Faults: {X_mf.shape}")

Datasets chargés :
  VBL-VA001        : (400, 45)
  CWRU             : (12627, 15)
  CMAPSS           : (17731, 126)
  Mechanical Faults: (36000, 132)


In [10]:
import time
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import HuberRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

def split_by_source(X, y, meta, test_size=0.2, random_state=42):
    """Split stratifié par fichier source, avec auto‑ajustement du test_size."""
    file_groups = defaultdict(list)
    file_label  = {}
    for i, m in enumerate(meta):
        fid = m['unit_id']
        file_groups[fid].append(i)
        file_label[fid] = y[i]

    all_fids    = list(file_groups.keys())
    all_flabels = [file_label[f] for f in all_fids]

    # Nombre de classes
    n_classes = len(set(all_flabels))
    n_files = len(all_fids)
    # Taille minimale de test pour avoir au moins un fichier par classe
    min_test_size = n_classes / n_files
    if test_size < min_test_size:
        print(f"  ⚠ Ajustement test_size : {test_size:.2f} → {min_test_size:.2f} "
              f"(car {n_classes} classes et {n_files} fichiers)")
        test_size = min_test_size

    sss = StratifiedShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )
    tr_idx, te_idx = next(sss.split(all_fids, all_flabels))

    train_idx = [i for j in tr_idx for i in file_groups[all_fids[j]]]
    test_idx  = [i for j in te_idx for i in file_groups[all_fids[j]]]

    return (X[train_idx], X[test_idx],
            [y[i] for i in train_idx],
            [y[i] for i in test_idx])


def benchmark_classification(X, y, meta, dataset_name, test_size=0.2):
    """
    Benchmark des 5 modèles sur un dataset de classification.
    Retourne un DataFrame avec les métriques.
    """
    from sklearn.preprocessing import LabelEncoder
    print(f"\n{'='*55}")
    print(f"Benchmark CLASSIFICATION — {dataset_name}")
    print(f"{'='*55}")

    X_train, X_test, y_train, y_test = split_by_source(
        X, y, meta, test_size
    )
    print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

    # Encodage des labels pour tous les modèles (par simplicité)
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)

    # Normalisation pour modèles sensibles aux échelles
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)

    modeles = {
        "Decision Tree": DecisionTreeClassifier(
            max_depth=15, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=42, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=6,
            learning_rate=0.05, eval_metric='mlogloss',
            random_state=42, n_jobs=-1
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            max_iter=200, random_state=42,
            early_stopping=True, validation_fraction=0.1
        ),
    }

    results = []
    for nom, modele in modeles.items():
        # MLP et Decision Tree peuvent bénéficier de la normalisation
        use_scaled = nom in ["MLP", "Decision Tree"]
        Xtr = X_tr_sc if use_scaled else X_train
        Xte = X_te_sc if use_scaled else X_test

        # Pour tous les modèles, on utilise les labels encodés
        t0 = time.time()
        modele.fit(Xtr, y_train_enc)
        train_time = time.time() - t0

        t0 = time.time()
        y_pred_enc = modele.predict(Xte)
        infer_time = (time.time() - t0) / len(X_test) * 1000

        # Décodage pour les métriques
        y_pred = le.inverse_transform(y_pred_enc)

        acc  = accuracy_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        print(f"  {nom:20s} → Acc: {acc*100:.2f}%  "
              f"F1: {f1:.3f}  "
              f"Train: {train_time:.1f}s  "
              f"Infer: {infer_time:.3f}ms")

        results.append({
            "Modèle"        : nom,
            "Dataset"       : dataset_name,
            "Accuracy (%)"  : round(acc*100, 2),
            "F1-score"      : round(f1, 3),
            "Temps train (s)": round(train_time, 1),
            "Inférence (ms)": round(infer_time, 3),
        })

    return pd.DataFrame(results)


def benchmark_regression(X, y_rul, meta, dataset_name,
                          test_size=0.2):
    """
    Benchmark des 5 modèles sur la prédiction RUL (régression).
    """
    print(f"\n{'='*55}")
    print(f"Benchmark RÉGRESSION RUL — {dataset_name}")
    print(f"{'='*55}")

    # Split par moteur
    motor_groups = defaultdict(list)
    for i, m in enumerate(meta):
        motor_groups[m['unit_id']].append(i)

    all_motors  = list(motor_groups.keys())
    np.random.seed(42)
    np.random.shuffle(all_motors)
    split       = int(0.8 * len(all_motors))

    train_idx = [i for m in all_motors[:split] for i in motor_groups[m]]
    test_idx  = [i for m in all_motors[split:]  for i in motor_groups[m]]

    X_train = X[train_idx]
    X_test  = X[test_idx]
    y_train = [y_rul[i] for i in train_idx]
    y_test  = [y_rul[i] for i in test_idx]

    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)

    # Score NASA (métrique officielle asymétrique)
    def nasa_score(y_true, y_pred):
        errors = np.array(y_pred) - np.array(y_true)
        score  = np.where(
            errors < 0,
            np.exp(-errors/13) - 1,
            np.exp(errors/10)  - 1
        )
        return float(np.sum(score))

    modeles = {
        "Huber Regressor": HuberRegressor(max_iter=300),
        "Decision Tree"  : DecisionTreeRegressor(
            max_depth=10, random_state=42
        ),
        "Random Forest"  : RandomForestRegressor(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        "XGBoost"        : XGBRegressor(
            n_estimators=300, max_depth=6,
            learning_rate=0.05, random_state=42, n_jobs=-1
        ),
        "MLP"            : MLPRegressor(
            hidden_layer_sizes=(256, 128, 64),
            max_iter=300, random_state=42,
            early_stopping=True
        ),
    }

    results = []
    for nom, modele in modeles.items():
        use_scaled = nom in ["Huber Regressor", "MLP"]
        Xtr = X_tr_sc if use_scaled else X_train
        Xte = X_te_sc if use_scaled else X_test

        t0 = time.time()
        modele.fit(Xtr, y_train)
        train_time = time.time() - t0

        y_pred     = modele.predict(Xte)
        y_pred_cap = np.clip(y_pred, 0, 125)

        mae   = mean_absolute_error(y_test, y_pred_cap)
        rmse  = np.sqrt(mean_squared_error(y_test, y_pred_cap))
        r2    = r2_score(y_test, y_pred_cap)
        nasa  = nasa_score(y_test, y_pred_cap)
        infer = (time.time() - t0) / len(X_test) * 1000

        print(f"  {nom:20s} → MAE: {mae:.2f}  "
              f"RMSE: {rmse:.2f}  "
              f"R²: {r2:.3f}  "
              f"NASA: {nasa:.0f}  "
              f"Train: {train_time:.1f}s")

        results.append({
            "Modèle"         : nom,
            "Dataset"        : dataset_name,
            "MAE (cycles)"   : round(mae, 2),
            "RMSE (cycles)"  : round(rmse, 2),
            "R²"             : round(r2, 3),
            "Score NASA"     : round(nasa, 0),
            "Temps train (s)": round(train_time, 1),
        })

    return pd.DataFrame(results)

In [12]:
# Nettoyage des NaN dans CMAPSS
print("NaN dans X_cmapss avant nettoyage :", np.isnan(X_cmapss).sum())
X_cmapss = np.nan_to_num(X_cmapss, nan=0.0)
print("NaN après nettoyage :", np.isnan(X_cmapss).sum())

NaN dans X_cmapss avant nettoyage : 119362
NaN après nettoyage : 0


In [13]:
results_classif = []
results_regress = []


# ── Classification ────────────────────────────────────────────────────────
for name, X, y, meta in [
    ("VBL-VA001",         X_vbl,   y_vbl,   meta_vbl),
    ("CWRU",              X_cwru,  y_cwru,  meta_cwru),
    ("Mechanical Faults", X_mf,    y_mf,    meta_mf),
]:
    df = benchmark_classification(X, y, meta, name)
    results_classif.append(df)

# ── Régression RUL (CMAPSS) ───────────────────────────────────────────────
rul_values = [m['rul'] for m in meta_cmapss]
df_rul = benchmark_regression(X_cmapss, rul_values, meta_cmapss, "CMAPSS")
results_regress.append(df_rul)

# ── Consolidation ─────────────────────────────────────────────────────────
df_classif_all = pd.concat(results_classif, ignore_index=True)
df_regress_all = pd.concat(results_regress, ignore_index=True)

print("\n\n" + "="*55)
print("TABLEAU BENCHMARK — CLASSIFICATION")
print("="*55)
print(df_classif_all.to_string(index=False))

print("\n\n" + "="*55)
print("TABLEAU BENCHMARK — RÉGRESSION RUL")
print("="*55)
print(df_regress_all.to_string(index=False))


Benchmark CLASSIFICATION — VBL-VA001
Train : 320 | Test : 80
  Decision Tree        → Acc: 100.00%  F1: 1.000  Train: 0.0s  Infer: 0.008ms
  Random Forest        → Acc: 100.00%  F1: 1.000  Train: 1.4s  Infer: 2.143ms
  XGBoost              → Acc: 100.00%  F1: 1.000  Train: 1.5s  Infer: 0.060ms
  MLP                  → Acc: 100.00%  F1: 1.000  Train: 0.4s  Infer: 0.019ms

Benchmark CLASSIFICATION — CWRU
  ⚠ Ajustement test_size : 0.20 → 0.25 (car 10 classes et 40 fichiers)
Train : 9,349 | Test : 3,278
  Decision Tree        → Acc: 93.93%  F1: 0.930  Train: 0.3s  Infer: 0.001ms
  Random Forest        → Acc: 97.38%  F1: 0.972  Train: 1.9s  Infer: 0.052ms
  XGBoost              → Acc: 97.59%  F1: 0.975  Train: 8.4s  Infer: 0.087ms
  MLP                  → Acc: 93.50%  F1: 0.926  Train: 14.3s  Infer: 0.007ms

Benchmark CLASSIFICATION — Mechanical Faults
Train : 28,800 | Test : 7,200
  Decision Tree        → Acc: 50.89%  F1: 0.424  Train: 9.0s  Infer: 0.000ms
  Random Forest        → Acc: 7

c:\Users\HP PRO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


  Huber Regressor      → MAE: 12.44  RMSE: 16.02  R²: 0.853  NASA: 14691  Train: 14.6s
  Decision Tree        → MAE: 12.70  RMSE: 18.53  R²: 0.804  NASA: 45826  Train: 2.0s
  Random Forest        → MAE: 10.50  RMSE: 14.00  R²: 0.888  NASA: 13255  Train: 69.8s
  XGBoost              → MAE: 9.61  RMSE: 13.35  R²: 0.898  NASA: 12249  Train: 18.9s
  MLP                  → MAE: 10.15  RMSE: 14.22  R²: 0.884  NASA: 13098  Train: 60.5s


TABLEAU BENCHMARK — CLASSIFICATION
       Modèle           Dataset  Accuracy (%)  F1-score  Temps train (s)  Inférence (ms)
Decision Tree         VBL-VA001        100.00     1.000              0.0           0.008
Random Forest         VBL-VA001        100.00     1.000              1.4           2.143
      XGBoost         VBL-VA001        100.00     1.000              1.5           0.060
          MLP         VBL-VA001        100.00     1.000              0.4           0.019
Decision Tree              CWRU         93.93     0.930              0.3           0.

In [ ]:
import pickle
from pathlib import Path

Path('../models').mkdir(exist_ok=True)

# ── Réentraîner les meilleurs modèles avec tout le dataset ────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.neural_network import MLPClassifier

print("Entraînement des modèles finaux sur 100% des données...\n")

# ── VBL-VA001 — XGBoost (100%) ────────────────────────────────────────────
le_vbl = LabelEncoder()
y_vbl_enc = le_vbl.fit_transform(y_vbl)
xgb_vbl = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
xgb_vbl.fit(X_vbl, y_vbl_enc)
with open('../models/xgb_vbl.pkl', 'wb') as f:
    pickle.dump({'model': xgb_vbl, 'label_encoder': le_vbl}, f)
print("✅ XGBoost VBL-VA001 sauvegardé")

# ── CWRU — XGBoost (97.59%) ───────────────────────────────────────────────
le_cwru = LabelEncoder()
y_cwru_enc = le_cwru.fit_transform(y_cwru)
xgb_cwru = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
xgb_cwru.fit(X_cwru, y_cwru_enc)
with open('../models/xgb_cwru.pkl', 'wb') as f:
    pickle.dump({'model': xgb_cwru, 'label_encoder': le_cwru}, f)
print("✅ XGBoost CWRU sauvegardé")

# ── Mechanical Faults — MLP (89.64%) ─────────────────────────────────────
scaler_mf = StandardScaler()
X_mf_sc   = scaler_mf.fit_transform(X_mf)
X_mf_sc = np.nan_to_num(X_mf_sc, nan=0.0, posinf=0.0, neginf=0.0)

mlp_mf    = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    max_iter=300, random_state=42,
    early_stopping=True, validation_fraction=0.1
)
mlp_mf.fit(X_mf_sc, y_mf)
with open('../models/mlp_mechanical_faults.pkl', 'wb') as f:
    pickle.dump({
        'model'  : mlp_mf,
        'scaler' : scaler_mf,
        'classes': mlp_mf.classes_
    }, f)
print("✅ MLP Mechanical Faults sauvegardé")

# ── CMAPSS — XGBoost RUL (MAE 9.61) ──────────────────────────────────────
rul_all = [m['rul'] for m in meta_cmapss]
xgb_rul = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    random_state=42, n_jobs=-1
)
xgb_rul.fit(X_cmapss, rul_all)
with open('../models/xgb_rul_cmapss.pkl', 'wb') as f:
    pickle.dump({'model': xgb_rul}, f)
print("✅ XGBoost RUL CMAPSS sauvegardé")

# ── Tous les modèles benchmark (pour comparaison dans IA Lab) ─────────────
from sklearn.linear_model import HuberRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

all_models = {
    'classification': {
        'VBL': {
            'Decision Tree': DecisionTreeClassifier(max_depth=15, random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
            'XGBoost'      : XGBClassifier(n_estimators=300, eval_metric='mlogloss', random_state=42, n_jobs=-1),
            'MLP'          : MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=200, random_state=42),
        },
        'CWRU': {
            'Decision Tree': DecisionTreeClassifier(max_depth=15, random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
            'XGBoost'      : XGBClassifier(n_estimators=300, eval_metric='mlogloss', random_state=42, n_jobs=-1),
            'MLP'          : MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=200, random_state=42),
        },
    },
    'regression': {
        'CMAPSS': {
            'Huber'        : HuberRegressor(max_iter=300),
            'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
            'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
            'XGBoost'      : XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1),
            'MLP'          : MLPRegressor(hidden_layer_sizes=(256,128,64), max_iter=300, random_state=42),
        }
    }
}

# Entraînement rapide de tous pour le benchmark IA Lab
print("\nEntraînement benchmark complet pour IA Lab...")
from sklearn.preprocessing import StandardScaler

scaler_vbl   = StandardScaler().fit(X_vbl)
scaler_cwru  = StandardScaler().fit(X_cwru)
scaler_cmapss= StandardScaler().fit(X_cmapss)

rul_cmapss = [m['rul'] for m in meta_cmapss]
le_vbl2    = LabelEncoder().fit(y_vbl)
le_cwru2   = LabelEncoder().fit(y_cwru)

trained = {
    'VBL'   : {},
    'CWRU'  : {},
    'CMAPSS': {}
}

for name, model in all_models['classification']['VBL'].items():
    X_sc = scaler_vbl.transform(X_vbl) if name == 'MLP' else X_vbl
    y_enc = le_vbl2.transform(y_vbl)
    model.fit(X_sc, y_enc)
    trained['VBL'][name] = model

for name, model in all_models['classification']['CWRU'].items():
    X_sc = scaler_cwru.transform(X_cwru) if name == 'MLP' else X_cwru
    y_enc = le_cwru2.transform(y_cwru)
    model.fit(X_sc, y_enc)
    trained['CWRU'][name] = model

for name, model in all_models['regression']['CMAPSS'].items():
    X_sc = scaler_cmapss.transform(X_cmapss) if name in ['Huber','MLP'] else X_cmapss
    model.fit(X_sc, rul_cmapss)
    trained['CMAPSS'][name] = model

# Sauvegarde du registre complet
with open('../models/benchmark_registry.pkl', 'wb') as f:
    pickle.dump({
        'models'   : trained,
        'scalers'  : {
            'VBL'   : scaler_vbl,
            'CWRU'  : scaler_cwru,
            'CMAPSS': scaler_cmapss,
            'MF'    : scaler_mf
        },
        'encoders' : {
            'VBL' : le_vbl2,
            'CWRU': le_cwru2,
            'MF'  : {'classes': mlp_mf.classes_}
        },
        'benchmark_results': {
            'classification': df_classif_all.to_dict(),
            'regression'    : df_regress_all.to_dict()
        }
    }, f)

print("✅ Registre complet sauvegardé → models/benchmark_registry.pkl")
print()
print("=== BILAN SEMAINE 2 — VALIDÉE ===")
print()
print("Modèles sauvegardés :")
print("  xgb_vbl.pkl               VBL-VA001    100%")
print("  xgb_cwru.pkl              CWRU          97.59%")
print("  mlp_mechanical_faults.pkl Mech. Faults  89.64%")
print("  xgb_rul_cmapss.pkl        CMAPSS RUL    MAE=9.61")
print("  benchmark_registry.pkl    Tous modèles  IA Lab")

Entraînement des modèles finaux sur 100% des données...

✅ XGBoost VBL-VA001 sauvegardé
✅ XGBoost CWRU sauvegardé


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
# Sauvegarde pour le dashboard
df_classif_all.to_csv('../data/processed/benchmark_classification.csv',
                       index=False)
df_regress_all.to_csv('../data/processed/benchmark_regression.csv',
                       index=False)

print("✅ Benchmarks sauvegardés")
print()
print("=== RÉSUMÉ SEMAINE 2 ===")
print()

# Meilleur modèle par dataset
for dataset in df_classif_all['Dataset'].unique():
    sub = df_classif_all[df_classif_all['Dataset']==dataset]
    best = sub.loc[sub['Accuracy (%)'].idxmax()]
    print(f"  {dataset:25s} → {best['Modèle']:20s} "
          f"{best['Accuracy (%)']:.2f}%")

best_rul = df_regress_all.loc[df_regress_all['MAE (cycles)'].idxmin()]
print(f"  {'CMAPSS RUL':25s} → {best_rul['Modèle']:20s} "
      f"MAE={best_rul['MAE (cycles)']:.2f} cycles")